# NB04 — Hybrid Models with PCA and LDA (FIXED)

Leakage-safe nested stratified CV. Labels are numerically encoded for all classifiers to ensure compatibility with MLP/XGBoost. Dimensionality reduction is fitted strictly inside training folds. Checkpoints are saved after every model/seed.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, random, warnings, hashlib, platform, sys
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ROOT = Path(r"/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1")
DATA = ROOT / "01_DATA"
NOTEBOOKS = ROOT / "02_NOTEBOOKS"
RESULTS = ROOT / "03_RESULTS"
MODELS = ROOT / "04_MODELS"
FIGURES = ROOT / "05_FIGURES"
TABLES = ROOT / "06_TABLES"
LOGS = ROOT / "07_LOGS"
EXPORTS = ROOT / "08_EXPORTS"
FINAL_ZIP = ROOT / "09_FINAL_ZIP"

for p in [DATA, NOTEBOOKS, RESULTS, MODELS, FIGURES, TABLES, LOGS, EXPORTS, FINAL_ZIP]:
    p.mkdir(parents=True, exist_ok=True)

DATASET = DATA / "INIAP_Dataset.xlsx"
SEEDS = [2026, 2027, 2028]
OUTER_FOLDS = 5
INNER_FOLDS = 3
TARGET = "Class"

assert DATASET.exists(), f"Dataset not found: {DATASET}"
print("ROOT:", ROOT)
print("DATASET:", DATASET)


In [ ]:

!pip -q install xgboost


In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier


In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, matthews_corrcoef, cohen_kappa_score,
    log_loss
)
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
import time

def metrics_dict(y_true, y_pred, y_prob=None):
    d = {
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }
    if y_prob is not None:
        try:
            d["log_loss"] = log_loss(y_true, y_prob)
        except Exception:
            d["log_loss"] = np.nan
    return d

def set_seed_if_supported(estimator, seed):
    est = clone(estimator)
    params = est.get_params(deep=True)
    updates = {}
    if "random_state" in params:
        updates["random_state"] = seed
    if "clf__random_state" in params:
        updates["clf__random_state"] = seed
    if updates:
        est.set_params(**updates)
    return est

def nested_cv_evaluate(name, estimator, param_grid, X, y, seed):
    outer = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=seed)
    rows, preds, params = [], [], []
    for fold, (tr, te) in enumerate(outer.split(X, y), start=1):
        print(f"  {name} | seed {seed} | outer fold {fold}/{OUTER_FOLDS}")
        inner = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=seed + fold)
        est = set_seed_if_supported(estimator, seed + fold)
        search = GridSearchCV(
            est,
            param_grid=param_grid,
            scoring="f1_macro",
            cv=inner,
            n_jobs=-1,
            refit=True,
            return_train_score=False,
            error_score="raise",
        )
        t0 = time.time()
        search.fit(X.iloc[tr], y.iloc[tr])
        elapsed = time.time() - t0
        best = search.best_estimator_
        pred = best.predict(X.iloc[te])
        prob = best.predict_proba(X.iloc[te]) if hasattr(best, "predict_proba") else None
        met = metrics_dict(y.iloc[te], pred, prob)
        met.update({
            "model": name,
            "seed": seed,
            "outer_fold": fold,
            "n_test": len(te),
            "fit_seconds": elapsed,
        })
        rows.append(met)
        params.append({
            "model": name,
            "seed": seed,
            "outer_fold": fold,
            "best_score_inner": search.best_score_,
            "fit_seconds": elapsed,
            "best_params": json.dumps(search.best_params_),
        })
        for j, idx in enumerate(te):
            rec = {
                "row_id": int(idx),
                "y_true": int(y.iloc[idx]),
                "y_pred": int(pred[j]),
                "model": name,
                "seed": seed,
                "outer_fold": fold,
            }
            if prob is not None:
                for k, cls in enumerate(best.classes_):
                    rec[f"prob_{int(cls)}"] = float(prob[j, k])
            preds.append(rec)
    return pd.DataFrame(rows), pd.DataFrame(preds), pd.DataFrame(params)


In [ ]:
OUT = RESULTS / "NB04_HYBRIDS"; OUT.mkdir(exist_ok=True)
TAB = TABLES / "NB04_HYBRIDS"; TAB.mkdir(exist_ok=True)
LOG = LOGS / "NB04_HYBRIDS"; LOG.mkdir(exist_ok=True)

# Clean analytical data and encode the four cultivar labels once for ALL models.
df = pd.read_excel(DATASET).rename(columns={"AspectRation": "AspectRatio"})
X = df.drop(columns=[TARGET]).copy()
y_text = df[TARGET].astype(str)
le = LabelEncoder()
y = pd.Series(le.fit_transform(y_text), index=y_text.index, name=TARGET)
class_map = dict(enumerate(le.classes_))
print("Class mapping:", class_map)

hybrids = {}
for dr_name, dr_obj, dr_grid in [
    ("PCA", PCA(), [2, 3, 4, 5, 6, 8, 10]),
    ("LDA", LinearDiscriminantAnalysis(), [1, 2, 3]),
]:
    hybrids[f"{dr_name}_SVM"] = (
        Pipeline([
            ("scale", StandardScaler()),
            ("dr", dr_obj),
            ("clf", SVC(kernel="rbf", probability=True)),
        ]),
        {
            "dr__n_components": dr_grid,
            "clf__C": [1, 10, 100],
            "clf__gamma": ["scale", 0.01, 0.1],
        },
    )
    hybrids[f"{dr_name}_MLP"] = (
        Pipeline([
            ("scale", StandardScaler()),
            ("dr", dr_obj),
            ("clf", MLPClassifier(max_iter=1200, early_stopping=True)),
        ]),
        {
            "dr__n_components": dr_grid,
            "clf__hidden_layer_sizes": [(64,), (128, 64)],
            "clf__alpha": [1e-4, 1e-3],
        },
    )
    hybrids[f"{dr_name}_XGBoost"] = (
        Pipeline([
            ("scale", StandardScaler()),
            ("dr", dr_obj),
            ("clf", XGBClassifier(
                objective="multi:softprob",
                eval_metric="mlogloss",
                n_jobs=1,
                tree_method="hist",
            )),
        ]),
        {
            "dr__n_components": dr_grid,
            "clf__n_estimators": [300, 600],
            "clf__max_depth": [3, 5],
            "clf__learning_rate": [0.03, 0.1],
        },
    )


In [ ]:
all_metrics, all_preds, all_params = [], [], []

for seed in SEEDS:
    for name, (est, grid) in hybrids.items():
        print(f"\n=== Running {name} | seed {seed} ===")
        m, p, b = nested_cv_evaluate(name, est, grid, X, y, seed)

        # Decode labels for human-readable output.
        p["y_true"] = p["y_true"].map(class_map)
        p["y_pred"] = p["y_pred"].map(class_map)
        rename_probs = {f"prob_{i}": f"prob_{cls}" for i, cls in class_map.items() if f"prob_{i}" in p.columns}
        p = p.rename(columns=rename_probs)

        all_metrics.append(m)
        all_preds.append(p)
        all_params.append(b)

        # Per-model/seed checkpoints: completed work is not lost if a later model fails.
        safe_name = name.replace("/", "_")
        m.to_csv(OUT / f"checkpoint_{safe_name}_seed{seed}_metrics.csv", index=False)
        p.to_csv(OUT / f"checkpoint_{safe_name}_seed{seed}_predictions.csv", index=False)
        b.to_csv(OUT / f"checkpoint_{safe_name}_seed{seed}_params.csv", index=False)
        print(f"Completed {name} | seed {seed}. Checkpoint saved.")

metrics = pd.concat(all_metrics, ignore_index=True)
preds = pd.concat(all_preds, ignore_index=True)
params = pd.concat(all_params, ignore_index=True)

metrics.to_csv(OUT / "hybrid_metrics_by_fold.csv", index=False)
preds.to_csv(OUT / "hybrid_oof_predictions.csv", index=False)
params.to_csv(OUT / "hybrid_best_params.csv", index=False)

summary = metrics.groupby("model").agg(
    mean_macro_f1=("f1_macro", "mean"),
    sd_macro_f1=("f1_macro", "std"),
    mean_accuracy=("accuracy", "mean"),
    mean_balanced_accuracy=("balanced_accuracy", "mean"),
    mean_mcc=("mcc", "mean"),
    mean_kappa=("kappa", "mean"),
    mean_fit_seconds=("fit_seconds", "mean"),
).sort_values("mean_macro_f1", ascending=False)
summary.to_csv(TAB / "hybrid_summary.csv")
display(summary)
